# Pass 3: Actions Demanded
LLM coding of actions called for in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Emotions (Passes 1–2) explain *how* the public feels; Pass 3 captures
*what they want done* — the actionable demand layer (investigate, resign,
compensate, reform, punish, "None"/implicit).

### Approach
Same chunked protocol with `../prompts/pass-03-actions-demanded.md`.
Single string `action_demanded` per comment; "None" or implicit-demand
description when no explicit call.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1–4** | Same as Pass 1 | Load → slim/clean → chunk (15) → LLM call + `json.loads` |
| **5. Export** | Save + inspect | `sample_pass03_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *Which emotions (Passes 1–2) predict which demands?*
- *Do demands escalate across phases (onset → lingering)?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass03_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import random

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 200
chunk_size = 5
sample_seed = 42
model = "gemma-4-31b-it"
overwrite = False

In [ ]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
OUTPUT_PATH = f"../output/{crisis_name}/sample_pass03_results.jsonl"
LOG_PATH    = f"../output/{crisis_name}/sample_pass03_failures.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-4-31b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 8192  # max output ceiling; model uses only what it needs
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)

## Micro-sample ingest & preprocess
Load `n_sample` rows from `INPUT_PATH`, slim to `video_id` + `text`,
drop nulls, strip whitespace. Keeps `row_id` for lossless merge at inspect time.

In [ ]:
full_df = pd.read_csv(INPUT_PATH)
print(f"Full phase rows: {len(full_df)}")

sample_df = full_df.head(n_sample).copy()  # deterministic for review; n_sample=200
sample_df = sample_df[["video_id", "text"]]
sample_df = sample_df.dropna()
sample_df["text"] = sample_df["text"].astype(str).str.strip()
sample_df = sample_df[sample_df["text"] != ""].reset_index(drop=True)
sample_df["row_id"] = sample_df.index
print(f"Sample rows: {len(sample_df)}")
sample_df.head()

## Batch chunking
Slice the sample into 15-row chunks and convert each chunk to a raw CSV string
for one LLM call apiece.

In [ ]:
chunks = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
chunk_csvs = [c[["video_id", "text"]].to_csv(index=False) for c in chunks]
print(f"{len(chunks)} chunks x ~{BATCH_SIZE} rows | first chunk chars: {len(chunk_csvs[0])}")
print(chunk_csvs[0][:600])

## LLM inference
Pass the system instruction + one CSV chunk per call to the Google AI Studio API.
Sequential calls with a 1s pause; raw responses kept in memory for the parse step.

In [ ]:
prompt_template = Path("../prompts/pass-03-actions-demanded.txt").read_text(encoding="utf-8").strip()
print(prompt_template[:400])

def call_with_retry(full_input, tries=4):
    for a in range(tries):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=full_input,
                config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS),
            )
        except Exception as e:
            if a == tries - 1:
                raise
            time.sleep(2 ** a + random.uniform(0, 1))  # exp backoff + jitter

raw_responses, failures = [], []
for i, csv_str in enumerate(chunk_csvs):
    try:
        resp = call_with_retry(prompt_template + "\n\n" + csv_str)
        raw_responses.append(resp.text or "")
    except Exception as e:
        failures.append({"chunk": i, "error": repr(e)})
        raw_responses.append("")
    print(f"chunk {i + 1}/{len(chunk_csvs)}: {len(raw_responses[-1])} chars")
    time.sleep(1)

print(f"ok: {sum(1 for r in raw_responses if r)} | failed chunks: {len(failures)}")

## API probe (run once)
Single-row call to isolate model/key issues from chunk payload issues before the full loop.

In [ ]:
probe = prompt_template + "\n\n" + sample_df.head(1)[["video_id", "text"]].to_csv(index=False)
r = client.models.generate_content(
    model=MODEL,
    contents=probe,
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=2048),
)
print((r.text or "")[:1500])